# Kirkland Si[111]/Si[110] Convergent-Probe CBED

Simulate a silicon convergent-beam electron diffraction pattern at 100 keV and compare WP-MS, AS-MS, and F-MS internally.

The default records a single crystal thickness near 1000 Å, supports scattering angles to 300 mrad, and generates the combined publication figure. Set `ZONE_AXIS = "111"` or `ZONE_AXIS = "110"` below.


In [ ]:
%matplotlib widget

import os
import sys
from pathlib import Path
from time import perf_counter

# Set these before importing abTEM/CuPy/JAX in a fresh kernel.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", ".70")


def _repo_root_from_notebook():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / "notebooks").exists() and (base / "wide_angle_propagation").exists():
            return base
    return cwd


REPO_ROOT = _repo_root_from_notebook()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import abtem
import cupy
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from ase.build import bulk
from matplotlib.colors import LogNorm

from wide_angle_propagation.notebook_utils import (
    assert_scattering_support,
    diffraction_angle_axes_mrad,
    diffraction_pattern_numpy,
    load_cbed_results,
    make_kirkland_probe,
    masked_angle_crop,
    radial_integrated_profiles,
    save_cbed_results,
    simulate_fresnel_as_exit_only,
    simulate_wpm_exit_only,
    validate_cbed_results,
)
from wide_angle_propagation.propagation_methods import (
    angular_spectrum_propagation_kernel,
    energy2wavelength,
    fresnel_propagation_kernel,
)

abtem.config.set({"device": "gpu"})
abtem.config.set({"precision": "float64"})
jax.config.update("jax_enable_x64", True)

print(f"Repository root: {REPO_ROOT}")
print(f"CUDA_VISIBLE_DEVICES = {os.environ.get('CUDA_VISIBLE_DEVICES')}")


## Experiment setup

The full run uses 100 keV electrons and an aberration-free 8 mrad probe. The target thickness is represented by the nearest integer number of repeated silicon unit cells along the selected zone axis.


In [ ]:
# --- Runtime controls -------------------------------------------------------
DEBUG_MODE = False
LOAD_EXISTING_RESULTS = False
RECOMPUTE_SIMULATION = True
SAVE_RESULTS_NPZ = True
SAVE_FIGURES = True

ZONE_AXIS = "111"  # Set to "111" or "110".
if ZONE_AXIS not in {"111", "110"}:
    raise ValueError(f"Unsupported ZONE_AXIS={ZONE_AXIS!r}; expected '111' or '110'.")

axis_tag = f"si{ZONE_AXIS}"
simulation_mode_label = "static crystal"
RESULTS_FILENAME = f"cbed_kirkland_{axis_tag}_100kev_300mrad_1000A.npz"
PATTERN_FIGURE_BASENAME = f"cbed_kirkland_{axis_tag}_pattern_300mrad_1000A"

# --- Kirkland Fig. 7.20-style optics ---------------------------------------
energy = 100e3
TARGET_THICKNESSES_A = np.array([1000.0], dtype=float)
PROBE_SEMIANGLE_MRAD = 8.0
CS_MM = 0.0
MAX_SCATTERING_ANGLE_MRAD = 300.0

# --- Numerical controls -----------------------------------------------------
# At 100 keV, 0.05 A sampling gives a Nyquist angle above 300 mrad.
POTENTIAL_SAMPLING_A = 0.05
SLICES_PER_CELL = 32
WPM_N_BINS = 128
WPM_POWER_SPACING = 2.0
RADIAL_BIN_WIDTH_MRAD = 0.5
CBED_LOG_VMIN = 1e-8

# The archived Si notebook used this lateral Si[111] supercell for Kirkland-like CBED.
FULL_LATERAL_REPEATS = (12, 7)
DEBUG_LATERAL_REPEATS = (4, 3)
DEBUG_TARGET_THICKNESSES_A = np.array([50.0], dtype=float)

if DEBUG_MODE:
    lateral_repeats = DEBUG_LATERAL_REPEATS
    target_thicknesses_A = DEBUG_TARGET_THICKNESSES_A.copy()
else:
    lateral_repeats = FULL_LATERAL_REPEATS
    target_thicknesses_A = TARGET_THICKNESSES_A.copy()

wavelength = float(energy2wavelength(energy))
cs_A = CS_MM * 1.0e7
scherzer_defocus_A = -1.2 * np.sqrt(cs_A * wavelength)

results_dir = REPO_ROOT / "notebooks" / "cbed" / "results"
results_path = results_dir / RESULTS_FILENAME
figures_dir = REPO_ROOT / "Paper" / "figures" / "method_explainer_generations"

print(f"Mode: {'debug' if DEBUG_MODE else 'full'}")
print(f"Zone axis: Si[{ZONE_AXIS}]")
print(f"Simulation mode: {simulation_mode_label}")
print(f"Energy: {energy / 1e3:.0f} keV, wavelength = {wavelength:.5f} A")
print(f"Probe aperture semi-angle: {PROBE_SEMIANGLE_MRAD:.1f} mrad")
print(f"Cs: {CS_MM:.2f} mm = {cs_A:.3e} A")
print(f"Scherzer defocus: {scherzer_defocus_A:.1f} A ({scherzer_defocus_A / 10:.1f} nm)")
print(f"Maximum scattering angle: {MAX_SCATTERING_ANGLE_MRAD:.0f} mrad")
print(f"Target thicknesses: {target_thicknesses_A} A")
print(f"Lateral repeats: {lateral_repeats[0]} x {lateral_repeats[1]}")
print(f"Result cache: {results_path}")



## Crystal setup

Construct an orthogonal diamond-cubic silicon unit cell using an explicit lattice constant. `Si[111]` uses the archived rotation recipe; `Si[110]` uses a rotated bulk-periodic cell so the beam direction remains periodic.



In [ ]:
SI_LATTICE_CONSTANT_A = 5.431
silicon = bulk("Si", "diamond", a=SI_LATTICE_CONSTANT_A, cubic=True)
if ZONE_AXIS == "111":
    zone_axis_label = "Si[111]"
    silicon_oriented = silicon.copy()
    silicon_oriented.rotate((0, 0, 1), (1, 1, 1), rotate_cell=True)
    silicon_oriented.rotate(45, "z", rotate_cell=True)
elif ZONE_AXIS == "110":
    zone_axis_label = "Si[110]"
    silicon_oriented = silicon.copy()
    silicon_oriented.rotate((0, 0, 1), (1, 1, 0), rotate_cell=True)
    silicon_oriented.rotate(45, "z", rotate_cell=True)
else:
    raise AssertionError(f"Unhandled ZONE_AXIS={ZONE_AXIS!r}")

unit_cell = abtem.orthogonalize_cell(silicon_oriented)
unit_cell.pbc = [True, True, True]
unit_cell.wrap()

nx_rep, ny_rep = lateral_repeats
unit_supercell = unit_cell * (nx_rep, ny_rep, 1)
unit_supercell.pbc = [True, True, True]

cell_lengths_A = np.array(unit_cell.cell.lengths(), dtype=float)
cell_thickness_A = float(unit_cell.cell[2, 2])
dz_test_A = cell_thickness_A / SLICES_PER_CELL
cell_repeats = np.maximum(1, np.rint(target_thicknesses_A / cell_thickness_A).astype(int))
actual_thicknesses_A = cell_repeats * cell_thickness_A
max_cell_repeats = int(cell_repeats.max())

print(f"Orthogonal {zone_axis_label} unit cell lengths: {cell_lengths_A} A")
print(f"{zone_axis_label} repeat thickness: {cell_thickness_A:.4f} A")
print(f"Slices per repeat: {SLICES_PER_CELL}, dz = {dz_test_A:.4f} A")
print("Thickness mapping:")
for target, repeat, actual in zip(target_thicknesses_A, cell_repeats, actual_thicknesses_A):
    print(f"  target {target:7.1f} A -> {repeat:4d} repeats -> actual {actual:8.2f} A")
print(f"One-cell lateral supercell atoms: {len(unit_supercell)}")
print(f"One-cell lateral field of view: {unit_supercell.cell.lengths()[0]:.2f} x {unit_supercell.cell.lengths()[1]:.2f} A")


## Simulation helpers

These helpers avoid storing every per-slice wavefront during the thick-specimen run. Only the exit wave after each repeated unit cell is retained, and CBED patterns are recorded at the requested thickness.


In [ ]:
METHOD_NAMES = ["WP-MS", "Angular Spectrum", "Fresnel MS"]
METHOD_LABELS = {
    "WP-MS": "WP-MS",
    "Angular Spectrum": "AS-MS",
    "Fresnel MS": "F-MS",
}
METHOD_COLORS = {
    "WP-MS": "C2",
    "Angular Spectrum": "C1",
    "Fresnel MS": "C0",
}
METHOD_LINESTYLES = {
    "WP-MS": "-",
    "Angular Spectrum": "--",
    "Fresnel MS": ":",
}
method_plot_order = METHOD_NAMES


def print_scattering_support(sampling_A):
    nyquist = assert_scattering_support(sampling_A, wavelength, MAX_SCATTERING_ANGLE_MRAD)
    print(f"Grid sampling: ({sampling_A[0]:.5f}, {sampling_A[1]:.5f}) A")
    print(f"Nyquist scattering angle: {nyquist:.1f} mrad")
    return nyquist


def build_potential_array(atoms, *, gpts=None, announce_label=None):
    potential_kwargs = {
        "slice_thickness": dz_test_A,
        "projection": "finite",
        "parametrization": "lobato",
    }
    if gpts is None:
        potential_kwargs["sampling"] = POTENTIAL_SAMPLING_A
    else:
        potential_kwargs["gpts"] = gpts

    potential_abtem = abtem.Potential(atoms, **potential_kwargs)
    gpts_local = tuple(int(v) for v in potential_abtem.gpts)
    sampling_local = tuple(float(v) for v in potential_abtem.sampling)
    built = cupy.asnumpy(potential_abtem.build(lazy=False).array)
    pot_local = jnp.array(built / dz_test_A, dtype=jnp.float64)
    del potential_abtem, built
    cupy.get_default_memory_pool().free_all_blocks()

    if announce_label is not None:
        print(f"Built potential for {announce_label}: shape={pot_local.shape}, grid={gpts_local}")
    if pot_local.shape[0] != SLICES_PER_CELL:
        print(
            "Warning: built potential has "
            f"{pot_local.shape[0]} slices, expected {SLICES_PER_CELL}. "
            "Using the configured dz for propagation."
        )
    return pot_local, gpts_local, sampling_local


def build_reference_potential_and_kernels():
    print(f"Building reference one-cell {zone_axis_label} potential...")
    pot_cell_local, gpts_local, sampling_local = build_potential_array(unit_supercell)
    fk_local = jnp.array(
        fresnel_propagation_kernel(*gpts_local, sampling_local, z=dz_test_A, energy=energy)
    )
    ak_local = jnp.array(
        angular_spectrum_propagation_kernel(*gpts_local, sampling_local, z=dz_test_A, energy=energy)
    )
    print(f"Potential shape: {pot_cell_local.shape}")
    print(f"Grid: {gpts_local}")
    print_scattering_support(sampling_local)
    return pot_cell_local, fk_local, ak_local, gpts_local, sampling_local


def metadata_dict(gpts, sampling_A, nyquist_mrad):
    return {
        "description": "Si CBED wide-angle propagation comparison",
        "energy_eV": float(energy),
        "wavelength_A": float(wavelength),
        "lattice_constant_A": float(SI_LATTICE_CONSTANT_A),
        "probe_semiangle_mrad": float(PROBE_SEMIANGLE_MRAD),
        "Cs_mm": float(CS_MM),
        "Cs_A": float(cs_A),
        "defocus_A": float(scherzer_defocus_A),
        "max_scattering_angle_mrad": float(MAX_SCATTERING_ANGLE_MRAD),
        "potential_sampling_requested_A": float(POTENTIAL_SAMPLING_A),
        "sampling_A": tuple(float(v) for v in sampling_A),
        "gpts": tuple(int(v) for v in gpts),
        "nyquist_angle_mrad": float(nyquist_mrad),
        "slices_per_cell": int(SLICES_PER_CELL),
        "wpm_n_bins": int(WPM_N_BINS),
        "wpm_power_spacing": float(WPM_POWER_SPACING),
        "potential_parametrization": "lobato",
        "potential_projection": "finite",
        "index_model": "kg",
        "ms_phase_convention": "paraxial_n2",
        "dz_A": float(dz_test_A),
        "cell_thickness_A": float(cell_thickness_A),
        "lateral_repeats": tuple(int(v) for v in lateral_repeats),
        "target_thicknesses_A": tuple(float(v) for v in target_thicknesses_A),
        "actual_thicknesses_A": tuple(float(v) for v in actual_thicknesses_A),
        "cell_repeats": tuple(int(v) for v in cell_repeats),
        "radial_bin_width_mrad": float(RADIAL_BIN_WIDTH_MRAD),
        "zone_axis": str(ZONE_AXIS),
        "zone_axis_label": str(zone_axis_label),
        "debug_mode": bool(DEBUG_MODE),
    }


## Load or run CBED simulation

The simulation reuses one static repeated-cell potential for efficiency and records the CBED pattern at the requested thickness.


In [ ]:
should_load = LOAD_EXISTING_RESULTS and results_path.exists() and not RECOMPUTE_SIMULATION

if should_load:
    cbed_results = load_cbed_results(results_path)
    gpts = tuple(cbed_results["gpts"])
    sampling = tuple(cbed_results["sampling_A"])
    theta_y_mrad = cbed_results["theta_y_mrad"]
    theta_x_mrad = cbed_results["theta_x_mrad"]
    print_scattering_support(sampling)
else:
    pot_cell, fk, ak, gpts, sampling = build_reference_potential_and_kernels()
    ny_g, nx_g = gpts
    theta_y_mrad, theta_x_mrad = diffraction_angle_axes_mrad(ny_g, nx_g, sampling, wavelength)

    probe = make_kirkland_probe(
        ny_g,
        nx_g,
        sampling,
        wavelength,
        PROBE_SEMIANGLE_MRAD,
        scherzer_defocus_A,
        cs_A,
    )
    jax.block_until_ready(probe)

    repeat_to_target_indices = {}
    for idx, repeat in enumerate(cell_repeats):
        repeat_to_target_indices.setdefault(int(repeat), []).append(idx)

    patterns = {
        method: np.zeros((len(cell_repeats), ny_g, nx_g), dtype=np.float64)
        for method in METHOD_NAMES
    }
    cell_runtime_s = {
        method: np.zeros(max_cell_repeats, dtype=float)
        for method in METHOD_NAMES
    }
    runtime_at_targets_s = {
        method: np.zeros(len(cell_repeats), dtype=float)
        for method in METHOD_NAMES
    }

    waves = {method: probe for method in METHOD_NAMES}
    cumulative_runtime_s = {method: 0.0 for method in METHOD_NAMES}
    print(f"Propagating {max_cell_repeats} {zone_axis_label} repeats through {METHOD_NAMES}...")

    for cell_idx in range(1, max_cell_repeats + 1):
        if cell_idx == 1 or cell_idx in repeat_to_target_indices or cell_idx % 10 == 0 or cell_idx == max_cell_repeats:
            print(f"  repeat {cell_idx:4d}/{max_cell_repeats} ({cell_idx * cell_thickness_A:8.2f} A)")

        for method in METHOD_NAMES:
            t0 = perf_counter()
            if method == "WP-MS":
                waves[method] = simulate_wpm_exit_only(
                    pot_cell,
                    waves[method],
                    dz_test_A,
                    energy,
                    sampling,
                    n_bins=WPM_N_BINS,
                    power_spacing=WPM_POWER_SPACING,
                    bin_batch_size=4,
                )
            elif method == "Angular Spectrum":
                waves[method] = simulate_fresnel_as_exit_only(pot_cell, waves[method], ak, dz_test_A, energy)
            elif method == "Fresnel MS":
                waves[method] = simulate_fresnel_as_exit_only(pot_cell, waves[method], fk, dz_test_A, energy)
            else:
                raise KeyError(method)
            jax.block_until_ready(waves[method])
            elapsed = perf_counter() - t0
            cell_runtime_s[method][cell_idx - 1] = elapsed
            cumulative_runtime_s[method] += elapsed

        if cell_idx in repeat_to_target_indices:
            for target_idx in repeat_to_target_indices[cell_idx]:
                for method in METHOD_NAMES:
                    patterns[method][target_idx] = diffraction_pattern_numpy(waves[method]).astype(np.float64)
                    runtime_at_targets_s[method][target_idx] = cumulative_runtime_s[method]

    patterns = {method: stack.astype(np.float32) for method, stack in patterns.items()}
    radial_bin_edges_mrad, radial_bin_centers_mrad, radial_sums, radial_profiles = radial_integrated_profiles(
        patterns,
        theta_y_mrad,
        theta_x_mrad,
        MAX_SCATTERING_ANGLE_MRAD,
        RADIAL_BIN_WIDTH_MRAD,
    )
    nyquist_mrad = print_scattering_support(sampling)

    cbed_results = {
        "method_names": METHOD_NAMES,
        "target_thicknesses_A": target_thicknesses_A.copy(),
        "actual_thicknesses_A": actual_thicknesses_A.copy(),
        "cell_repeats": cell_repeats.copy(),
        "gpts": gpts,
        "sampling_A": sampling,
        "theta_y_mrad": theta_y_mrad,
        "theta_x_mrad": theta_x_mrad,
        "radial_bin_edges_mrad": radial_bin_edges_mrad,
        "radial_bin_centers_mrad": radial_bin_centers_mrad,
        "patterns": patterns,
        "radial_sums": radial_sums,
        "radial_profiles": radial_profiles,
        "runtime_at_targets_s": runtime_at_targets_s,
        "cell_runtime_s": cell_runtime_s,
        "metadata": metadata_dict(gpts, sampling, nyquist_mrad),
    }

    if SAVE_RESULTS_NPZ:
        save_cbed_results(results_path, cbed_results)
        print(f"Saved compact results -> {results_path}")

    del fk, ak, pot_cell
    cupy.get_default_memory_pool().free_all_blocks()

print("Available result thicknesses:")
for target, repeat, actual in zip(
    cbed_results["target_thicknesses_A"],
    cbed_results["cell_repeats"],
    cbed_results["actual_thicknesses_A"],
):
    print(f"  target {target:7.1f} A -> {repeat:4d} repeats -> actual {actual:8.2f} A")

print("Timing at final recorded thickness:")
for method in cbed_results["method_names"]:
    print(f"  {METHOD_LABELS.get(method, method):<8s} {cbed_results['runtime_at_targets_s'][method][-1]:10.2f} s")


## Validate cached arrays

This is a lightweight check that the stored CBED patterns and radial profiles are finite and have the expected method/thickness dimensions.



In [ ]:
validate_cbed_results(cbed_results)
print("Validation passed: finite CBED patterns and radial profiles for all methods/thicknesses.")


## CBED pattern

Plot one CBED pattern at the largest recorded thickness, cropped to the configured scattering-angle limit.


In [ ]:
PLOT_METHOD = "WP-MS"
if PLOT_METHOD not in cbed_results["method_names"]:
    PLOT_METHOD = cbed_results["method_names"][0]

idx = int(np.argmax(cbed_results["actual_thicknesses_A"]))
target_A = cbed_results["target_thicknesses_A"][idx]
actual_A = cbed_results["actual_thicknesses_A"][idx]
repeat = int(cbed_results["cell_repeats"][idx])

crop, extent = masked_angle_crop(
    cbed_results["patterns"][PLOT_METHOD][idx],
    cbed_results["theta_y_mrad"],
    cbed_results["theta_x_mrad"],
    MAX_SCATTERING_ANGLE_MRAD,
)
finite = crop[np.isfinite(crop)]
if finite.size == 0 or float(finite.max()) <= 0.0:
    raise ValueError("CBED pattern has no positive intensity inside the plotting aperture")

display = crop / float(finite.max())
display = np.where(np.isfinite(display), np.maximum(display, CBED_LOG_VMIN), np.nan)

cmap = plt.get_cmap("gray").copy()
cmap.set_bad("black")

fig, ax = plt.subplots(figsize=(6.0, 5.6))
im = ax.imshow(
    display,
    origin="lower",
    extent=extent,
    cmap=cmap,
    norm=LogNorm(vmin=CBED_LOG_VMIN, vmax=1.0),
    interpolation="nearest",
)
ax.set_title(
    f"{METHOD_LABELS.get(PLOT_METHOD, PLOT_METHOD)} CBED, target {target_A:.0f} A, "
    f"actual {actual_A:.1f} A ({repeat:d} repeats)",
    fontsize=11,
)
ax.set_xlabel("theta_x (mrad)")
ax.set_ylabel("theta_y (mrad)")
ax.set_aspect("equal")
fig.suptitle(
    f"{zone_axis_label} CBED at {energy / 1e3:.0f} keV, {PROBE_SEMIANGLE_MRAD:.1f} mrad aperture, "
    f"{MAX_SCATTERING_ANGLE_MRAD:.0f} mrad crop",
    fontsize=12,
)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Normalized CBED intensity (log)")
fig.tight_layout()

if SAVE_FIGURES:
    figures_dir.mkdir(parents=True, exist_ok=True)
    png_path = figures_dir / f"{PATTERN_FIGURE_BASENAME}.png"
    pdf_path = figures_dir / f"{PATTERN_FIGURE_BASENAME}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved figure -> {png_path}")
    print(f"Saved figure -> {pdf_path}")

plt.show()


## Radially integrated intensity with peak finding

Radial profiles interpolated to a fine grid, with `scipy.signal.find_peaks` detecting Bragg rings on the log-transformed profile. Only the thickest specimen is shown.

In [ ]:
centers_raw = cbed_results["radial_bin_centers_mrad"]

# Interpolate onto a finer radial grid for smoother curves
FINE_BIN_WIDTH_MRAD = 0.1
centers = np.arange(0.0, centers_raw[-1] + FINE_BIN_WIDTH_MRAD, FINE_BIN_WIDTH_MRAD)

def resample_profile(profile_raw):
    """Linearly interpolate a radial profile onto the finer centers grid."""
    return np.interp(centers, centers_raw, np.asarray(profile_raw, dtype=np.float64))

idx = len(cbed_results["actual_thicknesses_A"]) - 1
target_A = cbed_results["target_thicknesses_A"][idx]
actual_A = cbed_results["actual_thicknesses_A"][idx]

fig, ax = plt.subplots(figsize=(9, 5.5))

# Peak-finding parameters
PEAK_MIN_ANGLE_MRAD = 5.0
PEAK_MAX_ANGLE_MRAD = 300.0
PEAK_PROMINENCE = 2   # in log10 units
PEAK_MIN_SPACING_MRAD = 4.0

dtheta = float(np.mean(np.diff(centers)))
min_peak_distance_bins = max(1, int(round(PEAK_MIN_SPACING_MRAD / dtheta)))
angle_mask = (centers >= PEAK_MIN_ANGLE_MRAD) & (centers <= PEAK_MAX_ANGLE_MRAD)

from scipy.signal import find_peaks

for method in method_plot_order:
    profile = resample_profile(cbed_results["radial_profiles"][method][idx])
    ax.semilogy(
        centers,
        np.maximum(profile, 1e-18),
        color=METHOD_COLORS.get(method, None),
        linestyle=METHOD_LINESTYLES.get(method, "-"),
        linewidth=1.8,
        label=METHOD_LABELS.get(method, method),
    )

    # Peak finding on log-transformed profile
    log_profile = np.log10(np.maximum(profile, 1e-18))
    masked = np.where(angle_mask, log_profile, -np.inf)
    peaks, props = find_peaks(
        masked, prominence=PEAK_PROMINENCE, distance=min_peak_distance_bins
    )
    print(f"{METHOD_LABELS.get(method, method):<8s}: {len(peaks):3d} peaks found")

    if len(peaks) > 0:
        ax.scatter(
            centers[peaks], profile[peaks],
            marker="o", s=40, facecolors="none",
            edgecolors=METHOD_COLORS.get(method, None),
            linewidths=1.2, zorder=5,
        )

ax.set_title(f"target {target_A:.0f} Å, actual {actual_A:.1f} Å — with detected peaks", fontsize=11)
ax.set_xlim(0.0, 300.0)
ax.set_ylim(1e-10, 1.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel("Scattering angle (mrad)")
ax.set_ylabel("Annular intensity / total intensity")
ax.legend(loc="best", frameon=False, fontsize=9)
fig.suptitle(f"Radially integrated {zone_axis_label} CBED intensity ({simulation_mode_label})", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# --- Peak shift comparison: Δθ vs reference peak angle ------------------------
# Match peaks between method pairs by nearest-neighbour and plot the shift.

MAX_MATCH_DISTANCE_MRAD = 4.0

method_pairs = [
    ("WP-MS", "Angular Spectrum", "WP-MS − AS-MS", "C2"),
    ("Fresnel MS", "Angular Spectrum", "F-MS − AS-MS", "C0"),
]

# Recompute peaks (same parameters as above)
all_peaks = {}
for method in method_plot_order:
    profile = resample_profile(cbed_results["radial_profiles"][method][idx])
    log_profile = np.log10(np.maximum(profile, 1e-18))
    masked = np.where(angle_mask, log_profile, -np.inf)
    peaks, _ = find_peaks(masked, prominence=PEAK_PROMINENCE, distance=min_peak_distance_bins)
    all_peaks[method] = centers[peaks]

fig, ax = plt.subplots(figsize=(9, 5))

fw_ref_angles = None
fw_shifts = None

for method_a, method_b, label, color in method_pairs:
    ref_peaks = all_peaks[method_b]
    cmp_peaks = all_peaks[method_a]

    shifts = []
    ref_angles = []
    for rp in ref_peaks:
        dists = np.abs(cmp_peaks - rp)
        j = np.argmin(dists)
        if dists[j] <= MAX_MATCH_DISTANCE_MRAD:
            ref_angles.append(rp)
            shifts.append(cmp_peaks[j] - rp)

    ref_angles = np.array(ref_angles)
    shifts = np.array(shifts)
    ax.scatter(ref_angles, shifts, s=30, color=color, label=label, alpha=0.85, zorder=4)
    if len(ref_angles) >= 2:
        ax.plot(ref_angles, shifts, color=color, linewidth=1.0, alpha=0.4)

    if method_a == "Fresnel MS" and method_b == "Angular Spectrum":
        fw_ref_angles = ref_angles.copy()
        fw_shifts = shifts.copy()

ax.axhline(0.0, color="0.5", linewidth=0.8)
ax.set_xlim(0.0, 300.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel("Reference peak angle (mrad)")
ax.set_ylabel("Peak shift Δθ (mrad)")
ax.set_title(f"Peak position shift — {actual_A:.0f} Å", fontsize=11)

# --- Parabolic vs exact dispersion relation error --------------------------
def dispersion_phase_error(angle_mrad, thickness_A, wavelength_A):
    """Phase error from the parabolic (Fresnel) vs. exact spherical dispersion.

    The exact propagator accumulates phase (2πt/λ)·[√(1−x) − 1]
    where  x = λ²q² = sin²θ  for a plane wave at scattering angle θ.

    The Fresnel propagator replaces √(1−x) with its Taylor expansion
    √(1−x) ≈ 1 − x/2, giving  φ_parab = (2πt/λ)·(−x/2).

    This function returns |φ_parab − φ_exact|, i.e. the truncation error
    of the square-root series (which grows ∝ x² ∝ θ⁴ at small angles).
    """
    theta = 1e-3 * np.asarray(angle_mrad, dtype=np.float64)
    x = np.sin(theta) ** 2                     # x = λ²q², 0 ≤ x < 1
    prefactor = 2.0 * np.pi / wavelength_A

    # Exact spherical dispersion:  √(1−x) − 1
    phase_exact = prefactor * (np.sqrt(1.0 - x) - 1.0)

    # Parabolic (Fresnel) dispersion:  Taylor  √(1−x) ≈ 1 − x/2  →  −x/2
    phase_parabolic = prefactor * (-0.5 * x)

    return np.abs(phase_parabolic - phase_exact)

if fw_ref_angles is not None and len(fw_ref_angles) >= 2:
    fw_ref = np.array(fw_ref_angles)
    fw_shifts_abs = np.abs(np.array(fw_shifts))
    theory_at_peaks = dispersion_phase_error(fw_ref, actual_A, wavelength)
    fit_scale = np.dot(theory_at_peaks, fw_shifts_abs) / np.dot(theory_at_peaks, theory_at_peaks)
    theory_angle = np.linspace(0, 300, 500)
    ax.plot(theory_angle, fit_scale * dispersion_phase_error(theory_angle, actual_A, wavelength),
            "k--", linewidth=1.5, alpha=0.7, label="scaled |Δφ| (parabolic − exact dispersion)")

ax.legend(loc="best", frameon=False, fontsize=8)
fig.suptitle(f"{zone_axis_label} CBED peak displacement ({simulation_mode_label})", fontsize=12)
fig.tight_layout()
plt.show()

## Combined publication figure

Three-panel layout: CBED pattern (WP-MS), radially integrated profiles with detected peaks, and peak displacement with scaled Fresnel-vs-exact phase-error curve.

In [ ]:
# --- Combined 3-panel publication figure ---------------------------------------
from matplotlib.gridspec import GridSpec

# (a) CBED pattern — WP-MS, thickest specimen ----------------------------------
crop, extent = masked_angle_crop(
    cbed_results["patterns"]["WP-MS"][idx],
    cbed_results["theta_y_mrad"],
    cbed_results["theta_x_mrad"],
    MAX_SCATTERING_ANGLE_MRAD,
)
finite = crop[np.isfinite(crop)]
cbed_img = crop / float(finite.max())
cbed_img = np.where(np.isfinite(cbed_img), np.maximum(cbed_img, CBED_LOG_VMIN), np.nan)

# (b) Radial profiles with peaks -----------------------------------------------
peak_data = {}
for method in method_plot_order:
    profile = resample_profile(cbed_results["radial_profiles"][method][idx])
    log_profile = np.log10(np.maximum(profile, 1e-18))
    masked = np.where(angle_mask, log_profile, -np.inf)
    peaks, _ = find_peaks(masked, prominence=PEAK_PROMINENCE, distance=min_peak_distance_bins)
    peak_data[method] = {"profile": profile, "peaks": peaks}

# (c) Peak shift data ----------------------------------------------------------
method_pairs_shift = [
    ("WP-MS", "Angular Spectrum", "WP-MS − AS-MS", "C2"),
    ("Fresnel MS", "Angular Spectrum", "F-MS − AS-MS", "C0"),
]
MAX_MATCH_DISTANCE_MRAD = 4.0

shift_data = {}
for method_a, method_b, label, color in method_pairs_shift:
    ref_peaks = all_peaks[method_b]
    cmp_peaks = all_peaks[method_a]
    ra, sh = [], []
    for rp in ref_peaks:
        dists = np.abs(cmp_peaks - rp)
        j = np.argmin(dists)
        if dists[j] <= MAX_MATCH_DISTANCE_MRAD:
            ra.append(rp)
            sh.append(cmp_peaks[j] - rp)
    shift_data[(method_a, method_b)] = {"ref_angles": np.array(ra), "shifts": np.array(sh),
                                         "label": label, "color": color}

# --- Build figure --------------------------------------------------------------
fig = plt.figure(figsize=(14, 7.5))
gs = GridSpec(2, 2, figure=fig, width_ratios=[1.2, 1.0], height_ratios=[1.0, 1.0],
              hspace=0.35, wspace=0.28,
              left=0.06, right=0.98, top=0.90, bottom=0.10)

# (a) CBED image — left column, spans both rows
ax_a = fig.add_subplot(gs[:, 0])
im_a = ax_a.imshow(cbed_img, origin="lower", extent=extent, cmap=cmap,
                    norm=LogNorm(vmin=CBED_LOG_VMIN, vmax=1.0), interpolation="nearest")
ax_a.set_title("WP-MS CBED pattern", fontsize=11, fontweight="bold", pad=6)
ax_a.set_title("(a)", loc="left", fontsize=11, fontweight="bold", pad=6)
ax_a.set_xlabel("$θ_x$ (mrad)")
ax_a.set_ylabel("$θ_y$ (mrad)")
ax_a.set_aspect("equal")
cbar_a = fig.colorbar(im_a, ax=ax_a, fraction=0.046, pad=0.04, shrink=0.80)
cbar_a.set_label("Norm. intensity (log)", fontsize=8)

# (b) Radial profiles + peaks — right top
ax_b = fig.add_subplot(gs[0, 1])
for method in method_plot_order:
    pd_m = peak_data[method]
    ax_b.semilogy(centers, np.maximum(pd_m["profile"], 1e-18),
                  color=METHOD_COLORS[method], linestyle=METHOD_LINESTYLES[method],
                  linewidth=1.5, label=METHOD_LABELS[method])
    if len(pd_m["peaks"]) > 0:
        ax_b.scatter(centers[pd_m["peaks"]], pd_m["profile"][pd_m["peaks"]],
                     marker="o", s=30, facecolors="none",
                     edgecolors=METHOD_COLORS[method], linewidths=1.0, zorder=5)
ax_b.set_title("Radial profiles with detected peaks", fontsize=11, fontweight="bold")
ax_b.set_title("(b)", loc="left", fontsize=11, fontweight="bold")
ax_b.set_xlim(0.0, 300.0)
ax_b.set_ylim(1e-10, 1.0)
ax_b.grid(True, alpha=0.3)
ax_b.set_xlabel("Scattering angle (mrad)")
ax_b.set_ylabel("Annular intensity / total intensity")
ax_b.legend(loc="lower left", frameon=False, fontsize=7.5)

# (c) Peak shift + theory — right bottom
ax_c = fig.add_subplot(gs[1, 1])
fw_ra, fw_sh = None, None
for (method_a, method_b), sd in shift_data.items():
    ax_c.scatter(sd["ref_angles"], sd["shifts"], s=28, color=sd["color"],
                 label=sd["label"], alpha=0.85, zorder=4)
    if len(sd["ref_angles"]) >= 2:
        ax_c.plot(sd["ref_angles"], sd["shifts"], color=sd["color"],
                  linewidth=0.8, alpha=0.35)
    if method_a == "Fresnel MS" and method_b == "Angular Spectrum":
        fw_ra, fw_sh = sd["ref_angles"], sd["shifts"]

if fw_ra is not None and len(fw_ra) >= 2:
    theory_at = dispersion_phase_error(np.array(fw_ra), actual_A, wavelength)
    fs = np.dot(theory_at, np.abs(np.array(fw_sh))) / np.dot(theory_at, theory_at)
    ta = np.linspace(0, 300, 500)
    ax_c.plot(ta, fs * dispersion_phase_error(ta, actual_A, wavelength),
              "k--", linewidth=1.3, alpha=0.65, label="scaled |Δφ| (parabolic − exact dispersion)")

ax_c.axhline(0.0, color="0.5", linewidth=0.8)
ax_c.set_title("Peak displacement with scaled phase error", fontsize=11, fontweight="bold")
ax_c.set_title("(c)", loc="left", fontsize=11, fontweight="bold")
ax_c.set_xlim(0.0, 300.0)
ax_c.grid(True, alpha=0.3)
ax_c.set_xlabel("Reference peak angle (mrad)")
ax_c.set_ylabel("Peak shift Δθ (mrad)")
ax_c.legend(loc="best", frameon=False, fontsize=7.5)

fig.suptitle(
    f"{zone_axis_label} CBED — {actual_A:.0f} Å at {energy/1e3:.0f} keV, "
    f"{PROBE_SEMIANGLE_MRAD:.0f} mrad probe",
    fontsize=13, fontweight="bold",
)

COMBINED_FIGURE_BASENAME = f"cbed_kirkland_{axis_tag}_combined_300mrad_{actual_A:.0f}A"

if SAVE_FIGURES:
    figures_dir.mkdir(parents=True, exist_ok=True)
    png_path = figures_dir / f"{COMBINED_FIGURE_BASENAME}.png"
    pdf_path = figures_dir / f"{COMBINED_FIGURE_BASENAME}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved figure -> {png_path}")
    print(f"Saved figure -> {pdf_path}")

plt.show()